# Method Comparison — Tree Oracle Simulator

Compare current methods on `tree_oracle_sim.json` outputs.
**Real-cost only** (`*_speedup_real`); ratio-mode columns are no longer produced.

**Methods (50% gap goal):**
- Baselines: `single:eagle3`, `single:suffix`
- Hybrid baselines: `hybrid_e3:t` (paper-faithful), `hybrid_e3_sfx:F:T:N:t` (custom suffix params, when present)
- Extension oracles: `extension_oracle`, `extension_prune_pt_oracle:t`, `extension_sfx_oracle:F:T:N`
- Deployable extension: `extension`, `extension_sfx:F:T:N`, `extension_prune_pt:t`, `extension_hybrid:t`

**Forbidden methods are filtered out** (per `project_50pct_gap_goal.md` rules — 2026-04-27):
- `extension_oracle_path`, `extension_hybrid_perfect_oracle*`
- `extension_*_nlevel*`, `extension_dual_method*`, `extension_pure_sfx*`
- `extension_dmsfx*`, `extension_by_*` (deprecated)
- These methods were removed from the simulator and are NOT displayed even when present in old JSONs.

Cross-config comparison: each method picks its OWN best (s, k, B, threshold) — never
constrained to a shared reslice (per `feedback_method_optimum_independent.md`).

In [ ]:
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# ========== CONFIG ==========
# Specify ONLY the workload. The notebook auto-finds each method's best
# (s, k, B, F, T, N, threshold) combo across ALL sims of that workload.

WORKLOAD = "swebench"     # e.g. "swebench", "specbench", "bfcl_v3", "lcc"

SIM_DIRS = [
    Path("../results/explorations"),
    Path("../results/qwen3_14b"),
    Path("../results/legacy/early_sims"),
]
MODEL_LATENCY = Path("../config/latency/qwen3_14b.json")

# Auto-discover sim JSONs
SIM_FILES = {}  # name -> (path, info_dict)

def _parse_filename(stem):
    info = {"workload": None, "steps": None, "topk": None, "budget": None}
    m = re.search(r"(swebench|specbench|bfcl_v[34]|longbench_lcc|longbench_repobench|lcc|repobench)", stem)
    if m: info["workload"] = m.group(1)
    m = re.search(r"s(\d+)k(\d+)", stem)
    if m: info["steps"] = int(m.group(1)); info["topk"] = int(m.group(2))
    return info

def _is_sim_json(path):
    try:
        with open(path) as f:
            d = json.load(f)
        return ("latency" in d and isinstance(d["latency"], dict)
                and "budget_sweep" in d["latency"])
    except Exception:
        return False

for d in SIM_DIRS:
    if not d.exists(): continue
    candidates = list(d.glob("*.json")) + list(d.glob("*/tree_oracle_sim.json"))
    for p in candidates:
        if not _is_sim_json(p): continue
        stem = p.parent.name if p.name == "tree_oracle_sim.json" else p.stem
        info = _parse_filename(stem)
        SIM_FILES[stem] = (str(p), info)

# Filter to selected workload
WORKLOAD_SIMS = {k: v for k, v in SIM_FILES.items()
                 if v[1]["workload"] == WORKLOAD}
if not WORKLOAD_SIMS:
    raise RuntimeError(f"No sim JSONs found for workload={WORKLOAD!r} under {SIM_DIRS}.\n"
                       f"Have workloads: {sorted({v[1]['workload'] for v in SIM_FILES.values()})}")

print(f"Workload: {WORKLOAD}")
print(f"Discovered sims ({len(WORKLOAD_SIMS)}):")
for k in sorted(WORKLOAD_SIMS):
    info = WORKLOAD_SIMS[k][1]
    sk = f"s={info['steps']} k={info['topk']}" if info['steps'] else "(no s/k tag)"
    print(f"  {k}  [{sk}]")

# Load latency config (vanilla_step_ms used by step-cost chart)
vanilla_ms = float(json.load(open(MODEL_LATENCY))["vanilla_step_ms"])
print(f"\nvanilla_step_ms = {vanilla_ms:.3f} ms (from {MODEL_LATENCY})")


## 1. Method Taxonomy

Classify each method into a family + flag whether it's allowed for the 50% gap claim.

In [ ]:
# Method classification: family + display label + hyperparam extractor + colors.
# Forbidden methods are filtered out by the parsing cell.
#
# FAIRNESS: every suffix-using method sweeps (F, T, N) and picks its best.
# So `extension_oracle` family ABSORBS `extension_sfx_oracle:F:T:N` variants,
# `extension` ABSORBS `extension_sfx:F:T:N`, `hybrid_e3` ABSORBS
# `hybrid_e3_sfx:F:T:N:t`. The chart picks the best (F, T, N) per family
# AND the best (s, k, B) across all sims of the workload.

FORBIDDEN_PATTERNS = [
    r"^extension_oracle_path$",
    r"^extension_hybrid_perfect_oracle",
    r"^extension_nlevel",
    r"^extension_2level",
    r"^extension_sfx_backbone",
    r"^extension_anchor",
    r"^extension_hybrid_prune_pt",
    r"^extension_dual_method",
    r"^extension_pure_sfx",
    r"^extension_dmsfx",
    r"^extension_by_pathprob",
    r"^extension_by_pt",
]

def is_forbidden(name):
    return any(re.match(p, name) for p in FORBIDDEN_PATTERNS)

def family(name):
    if name == "eagle3" or name.startswith("single:eagle3"):
        return "single:eagle3"
    if name == "suffix" or name.startswith("single:suffix"):
        return "single:suffix"
    if name == "draft_model" or name.startswith("single:draft_model"):
        return "single:draft_model"
    if name.startswith("hybrid_e3"):
        return "hybrid_e3"
    if name.startswith("hybrid_dm"):
        return "hybrid_dm"
    if name == "extension_oracle" or name.startswith("extension_sfx_oracle"):
        return "extension_oracle"
    if name == "extension" or name.startswith("extension_sfx"):
        return "extension"
    if name.startswith("extension_prune_pt_oracle"):
        return "extension_prune_pt_oracle"
    if name.startswith("extension_prune_pt"):
        return "extension_prune_pt"
    if name.startswith("extension_hybrid_oracle"):
        return "extension_hybrid_oracle"
    if name.startswith("extension_hybrid"):
        return "extension_hybrid"
    if name.startswith("extension_by_count_score"):
        return "extension_by_count_score"
    if name.startswith("extension_by_count"):
        return "extension_by_count"
    if name.startswith("extension_by_score"):
        return "extension_by_score"
    return "other"

PRETTY_LABEL = {
    "single:eagle3":              "EAGLE3",
    "single:suffix":              "Suffix",
    "single:draft_model":         "Draft LM",
    "hybrid_e3":                  "Hybrid (Sfx→E3)",
    "hybrid_dm":                  "Hybrid (Sfx→DM)",
    "extension":                  "Extension",
    "extension_oracle":           "Extension Oracle ★",
    "extension_prune_pt":         "Extension (prune backbone)",
    "extension_prune_pt_oracle":  "Extension Oracle (prune backbone)",
    "extension_hybrid":           "Extension (suffix hybrid)",
    "extension_hybrid_oracle":    "Extension Oracle (suffix hybrid)",
    "extension_by_count":         "Extension (count cap)",
    "extension_by_score":         "Extension (score filter)",
    "extension_by_count_score":   "Extension (count+score)",
    "other":                      "?",
}

# Families whose result is INDEPENDENT of B (so don't show "B=" on bars).
BUDGET_INDEPENDENT_FAMS = {"single:suffix"}

FAMILY_COLOR = {
    "single:eagle3":              "#1f77b4",
    "single:suffix":              "#ffc107",
    "single:draft_model":         "#17becf",
    "hybrid_e3":                  "#9467bd",
    "hybrid_dm":                  "#e377c2",
    "extension":                  "#ff7f0e",
    "extension_oracle":           "#d62728",
    "extension_prune_pt":         "#ffbb70",
    "extension_prune_pt_oracle":  "#cc4040",
    "extension_hybrid":           "#ffcc99",
    "extension_hybrid_oracle":    "#aa3030",
    "extension_by_count":         "#bcbd22",
    "extension_by_score":         "#8c8c00",
    "extension_by_count_score":   "#a09d20",
    "other":                      "#cccccc",
}

FAMILY_ORDER = [
    "single:eagle3", "single:suffix", "single:draft_model",
    "hybrid_e3", "hybrid_dm",
    "extension", "extension_prune_pt", "extension_hybrid",
    "extension_by_count", "extension_by_score", "extension_by_count_score",
    "extension_oracle", "extension_prune_pt_oracle", "extension_hybrid_oracle",
]

DEFAULT_SUFFIX_HP = {
    "extension":                  "F=4.0 T=0.0 N=256",
    "extension_oracle":           "F=4.0 T=0.0 N=256",
    "hybrid_e3":                  "F=1.0 T=0.1",
    "hybrid_dm":                  "F=1.0 T=0.1",
    "extension_prune_pt":         "F=4.0 T=0.0 N=256",
    "extension_prune_pt_oracle":  "F=4.0 T=0.0 N=256",
    "extension_hybrid":           "F=4.0 T=0.0 N=256",
    "extension_hybrid_oracle":    "F=4.0 T=0.0 N=256",
}

def hyperparam_str(name):
    fam = family(name)
    parts = []
    if fam == "hybrid_e3":
        if name.startswith("hybrid_e3_sfx"):
            m = re.search(r"_f([\d.]+)", name);   _f = m.group(1) if m else None
            m = re.search(r"_t([\d.]+)_n", name); _t = m.group(1) if m else None
            m = re.search(r"_n(\d+)_", name);      _n = m.group(1) if m else None
            m = re.search(r"_th([\d.]+)", name);   _th = m.group(1) if m else None
            if _f: parts.append(f"F={_f}")
            if _t: parts.append(f"T={_t}")
            if _n: parts.append(f"N={_n}")
            if _th: parts.append(f"τ={_th}")
        else:
            m = re.search(r"_t([\d.]+)$", name)
            if m: parts.append(f"τ={m.group(1)}")
            parts.append(DEFAULT_SUFFIX_HP.get(fam, ""))
    elif fam == "hybrid_dm":
        m = re.search(r"_t([\d.]+)$", name)
        if m: parts.append(f"τ={m.group(1)}")
        parts.append(DEFAULT_SUFFIX_HP.get(fam, ""))
    elif fam == "extension":
        if name.startswith("extension_sfx"):
            m = re.search(r"_f([\d.]+)", name)
            if m: parts.append(f"F={m.group(1)}")
            m = re.search(r"_t([\d.]+)_n", name)
            if m: parts.append(f"T={m.group(1)}")
            m = re.search(r"_n(\d+)", name)
            if m: parts.append(f"N={m.group(1)}")
        else:
            parts.append(DEFAULT_SUFFIX_HP.get(fam, ""))
    elif fam == "extension_oracle":
        if name.startswith("extension_sfx_oracle"):
            m = re.search(r"_f([\d.]+)", name)
            if m: parts.append(f"F={m.group(1)}")
            m = re.search(r"_t([\d.]+)_n", name)
            if m: parts.append(f"T={m.group(1)}")
            m = re.search(r"_n(\d+)", name)
            if m: parts.append(f"N={m.group(1)}")
        else:
            parts.append(DEFAULT_SUFFIX_HP.get(fam, ""))
    elif fam in ("extension_prune_pt", "extension_prune_pt_oracle"):
        m = re.search(r"_t([\d.]+)$", name)
        if m: parts.append(f"pt={m.group(1)}")
        parts.append(DEFAULT_SUFFIX_HP.get(fam, ""))
    elif fam in ("extension_hybrid", "extension_hybrid_oracle"):
        m = re.search(r"_t([\d.]+)$", name)
        if m: parts.append(f"τ={m.group(1)}")
        parts.append(DEFAULT_SUFFIX_HP.get(fam, ""))
    elif fam == "extension_by_count":
        m = re.search(r"_r(\d+)$", name)
        if m: parts.append(f"r={m.group(1)}")
    elif fam == "extension_by_score":
        m = re.search(r"_t([\d.]+)$", name)
        if m: parts.append(f"s≥{m.group(1)}")
    elif fam == "extension_by_count_score":
        m1 = re.search(r"_r(\d+)", name)
        m2 = re.search(r"_t([\d.]+)$", name)
        if m1: parts.append(f"r={m1.group(1)}")
        if m2: parts.append(f"s≥{m2.group(1)}")
    return " ".join(p for p in parts if p)


def bar_sub_label(family_name, hp, best_budget, sk_tag=None):
    """Small text above each bar: B + reslice + hyperparams.
    For budget-independent families (suffix), skip B=. Reslice (sk_tag) shown
    if available so the viewer knows WHERE the per-method best came from."""
    parts = []
    if family_name not in BUDGET_INDEPENDENT_FAMS:
        parts.append(f"B={best_budget}")
    if sk_tag and sk_tag != "?":
        parts.append(sk_tag)
    if hp:
        parts.append(hp)
    return "\n".join(parts) if parts else ""


## 2. Parse `budget_sweep` → DataFrame

In [ ]:
# Scan ALL sims for the workload, collect every method-variant measurement.
# Then per-family pick the variant with highest spd_real.
# Each method is allowed its OWN best (s, k, B, F, T, N, threshold).

# Per-sim DataFrames (used later by budget-sensitivity chart for full curves)
SIM_DFS = {}

all_records = []
for sim_name, (path, info) in WORKLOAD_SIMS.items():
    with open(path) as f:
        d = json.load(f)
    df_i = pd.DataFrame(d["latency"]["budget_sweep"])
    SIM_DFS[sim_name] = df_i
    s_tag = info["steps"]
    k_tag = info["topk"]
    sk = f"s={s_tag}k={k_tag}" if s_tag else "?"
    for col in df_i.columns:
        if not col.endswith("_speedup_real"): continue
        m_name = col[:-len("_speedup_real")]
        if m_name.endswith("_always"): continue
        if is_forbidden(m_name): continue
        spd_col = df_i[col].dropna()
        if spd_col.empty: continue
        for idx in df_i.index:
            spd = df_i[col].iloc[idx]
            if pd.isna(spd): continue
            B = int(df_i["budget"].iloc[idx])
            mat = float(df_i.get(f"{m_name}_mat", pd.Series([np.nan])).iloc[idx]) if f"{m_name}_mat" in df_i.columns else 0
            steps = int(df_i.get(f"{m_name}_steps", pd.Series([1])).iloc[idx]) or 1
            target_ms = float(df_i.get(f"{m_name}_total_target_ms", pd.Series([0])).iloc[idx]) / steps
            draft_ms = float(df_i.get(f"{m_name}_total_draft_ms", pd.Series([0])).iloc[idx]) / steps
            avg_ext_size = float(df_i.get(f"{m_name}_total_target_tokens", pd.Series([0])).iloc[idx]) / steps
            fam = family(m_name)
            all_records.append({
                "method": m_name,
                "family": fam,
                "pretty": PRETTY_LABEL.get(fam, fam),
                "sim": sim_name,
                "s": s_tag, "k": k_tag, "sk_tag": sk,
                "best_budget": B,
                "speedup_real": float(spd),
                "mat": mat,
                "avg_target_ms": target_ms,
                "avg_draft_ms": draft_ms,
                "avg_step_ms": target_ms + draft_ms,
                "avg_ext_size": avg_ext_size,
            })

mdf_all = pd.DataFrame(all_records)
print(f"Total measurements scanned: {len(mdf_all)} across {len(WORKLOAD_SIMS)} sims")

# Per-FAMILY best across ALL sims/budgets/variants — each family picks its OWN
# (s, k, B, F, T, N, threshold) optimum. This is the fair "method-optimum" view.
mdf_best = (mdf_all.sort_values("speedup_real", ascending=False)
                   .drop_duplicates(subset="family")
                   .reset_index(drop=True))
mdf_best["color"] = mdf_best["family"].map(FAMILY_COLOR).fillna("#cccccc")
mdf_best["hp"] = mdf_best["method"].map(hyperparam_str)

# Apply display order
mdf_best["order"] = mdf_best["family"].map({f: i for i, f in enumerate(FAMILY_ORDER)})
mdf_best = mdf_best.dropna(subset=["order"]).sort_values("order").reset_index(drop=True)
print(f"Per-family best (cross-sim): {len(mdf_best)} entries")
mdf_best[["pretty", "method", "sim", "sk_tag", "best_budget", "hp", "speedup_real", "mat"]]


## 3. Summary Table — best of each family

★ = headline (allowed); ⚠ = forbidden by 50% goal rules

In [ ]:
print(f"=== {WORKLOAD} — best per method (each method picks its OWN s, k, B, F, T, N, t) ===")
print(f"{'':2} {'method':<35} {'variant':<46} {'reslice':<10} {'B':>4} {'spd_real':>9} {'mat':>6}")
print("-" * 125)
for _, r in mdf_best.iterrows():
    mark = "★" if r["family"] == "extension_oracle" else " "
    print(f"{mark:2} {r['pretty']:<35} {r['method'][:44]:<46} {r['sk_tag']:<10} {r['best_budget']:>4} "
          f"{r['speedup_real']:>9.3f} {r['mat']:>6.2f}")

# Headline gap
ext_or = mdf_best[mdf_best["family"] == "extension_oracle"]
hyb_all = mdf_best[mdf_best["family"] == "hybrid_e3"]
ext_dep_fams = ["extension", "extension_prune_pt", "extension_hybrid",
                "extension_by_count", "extension_by_score", "extension_by_count_score"]
ext_dep = mdf_best[mdf_best["family"].isin(ext_dep_fams)]

if len(ext_or) and len(hyb_all):
    e = ext_or.iloc[0]
    h = hyb_all.sort_values("speedup_real", ascending=False).iloc[0]
    gap = (e["speedup_real"] / h["speedup_real"] - 1) * 100
    print(f"\n★ ORACLE GAP (each method's own optimum):")
    print(f"   ext: {e['pretty']}  ({e['method']})")
    print(f"        = {e['speedup_real']:.3f}× @ {e['sk_tag']} B={e['best_budget']} ({e['sim']})")
    print(f"   hyb: {h['pretty']}  ({h['method']})")
    print(f"        = {h['speedup_real']:.3f}× @ {h['sk_tag']} B={h['best_budget']} ({h['sim']})")
    print(f"   GAP: {gap:+.2f}%   (50% target: {50 - gap:+.2f}% remaining)")
if len(ext_dep) and len(hyb_all):
    e = ext_dep.sort_values("speedup_real", ascending=False).iloc[0]
    h = hyb_all.sort_values("speedup_real", ascending=False).iloc[0]
    gap_dep = (e["speedup_real"] / h["speedup_real"] - 1) * 100
    print(f"\n  DEPLOYABLE GAP:")
    print(f"   ext: {e['pretty']}  ({e['method']}) = {e['speedup_real']:.3f}× @ {e['sk_tag']} B={e['best_budget']}")
    print(f"   hyb: {h['pretty']}  ({h['method']}) = {h['speedup_real']:.3f}× @ {h['sk_tag']} B={h['best_budget']}")
    print(f"   GAP: {gap_dep:+.2f}%")


## 4. Speedup chart — best per method, with oracle↔hybrid gap arrow

Each method's best variant. Hyperparameters labeled above each bar.
Red arrow shows percentage gap from best `Hybrid` to `Ext-Oracle`.

In [ ]:
def _draw_gap_arrow(ax, mdf_plot, hyb_families, ext_families, offset=0.4):
    fams = list(mdf_plot["family"].values)
    spds = list(mdf_plot["speedup_real"].values)
    h_picks = [(spds[i], i) for i, f in enumerate(fams) if f in hyb_families]
    e_picks = [(spds[i], i) for i, f in enumerate(fams) if f in ext_families]
    if not h_picks or not e_picks:
        return None
    h_spd, h_idx = max(h_picks)
    e_spd, e_idx = max(e_picks)
    pct = (e_spd / h_spd - 1) * 100
    x_lo, x_hi = sorted([h_idx, e_idx])
    ax.hlines(h_spd, x_lo - 0.4, x_hi + offset, colors="red",
              linestyles="--", alpha=0.8, linewidth=1.5)
    ax.hlines(e_spd, x_lo - 0.4, x_hi + offset, colors="red",
              linestyles="--", alpha=0.8, linewidth=1.5)
    arrow_x = x_hi + offset + 0.2
    ax.annotate("", xy=(arrow_x, e_spd), xytext=(arrow_x, h_spd),
                arrowprops=dict(arrowstyle="<->", color="red", lw=2))
    ax.text(arrow_x + 0.15, (h_spd + e_spd) / 2, f"{pct:+.1f}%",
            ha="left", va="center", color="red", fontsize=11, fontweight="bold")
    return pct


def _draw_speedup_chart(mdf_plot, title, hyb_fams, ext_fams):
    fig, ax = plt.subplots(figsize=(max(11, len(mdf_plot) * 1.05), 7.5))
    x = np.arange(len(mdf_plot))
    ax.bar(x, mdf_plot["speedup_real"].values, color=mdf_plot["color"].values,
           edgecolor="black", linewidth=0.4)
    for i, r in mdf_plot.iterrows():
        spd = r["speedup_real"]
        marker = " ★" if r["family"] == "extension_oracle" else ""
        ax.text(i, spd + 0.05, f"{spd:.2f}×{marker}",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
        sub = bar_sub_label(r["family"], r["hp"], r["best_budget"], r.get("sk_tag"))
        if sub:
            ax.text(i, spd + 0.20, sub, ha="center", va="bottom",
                    fontsize=7, color="#444")
    pct = _draw_gap_arrow(ax, mdf_plot, hyb_fams, ext_fams)
    ax.set_xticks(x)
    ax.set_xticklabels(mdf_plot["pretty"], rotation=35, ha="right", fontsize=9)
    ax.set_ylabel("Real-cost speedup (×)")
    ax.set_title(title, fontsize=12)
    ax.axhline(y=1.0, color="red", linestyle=":", alpha=0.3)
    ax.grid(axis="y", alpha=0.2)
    ax.set_ylim(top=mdf_plot["speedup_real"].max() * 1.40)
    plt.tight_layout(); plt.show()
    return pct


HYB_FAMS = {"hybrid_e3"}
ORACLE_FAMS = {"extension_oracle"}
pct_or = _draw_speedup_chart(
    mdf_best, f"{WORKLOAD} — best per method (own s/k/B/F/T/N/τ)  ★=Oracle gap",
    HYB_FAMS, ORACLE_FAMS)
print(f"Oracle vs Hybrid gap: {pct_or:+.2f}%")


## 4b. Speedup chart — gap arrow for **deployable** extension vs hybrid

Same chart, but the red arrow now compares the best **deployable extension**
(no oracle accounting) against the best hybrid baseline. This is the
"realistic gap" you could ship today.

In [ ]:
DEPLOY_FAMS = {"extension", "extension_prune_pt", "extension_hybrid",
               "extension_by_count", "extension_by_score",
               "extension_by_count_score"}
pct_dep = _draw_speedup_chart(
    mdf_best, f"{WORKLOAD} — gap (Hybrid → best Deployable Extension)",
    HYB_FAMS, DEPLOY_FAMS)
print(f"Deployable Ext vs Hybrid gap: {pct_dep:+.2f}%")


## 5. MAT chart — best per method (matched to speedup-best variant)

Each method's **MAT** at the same variant that maxes its speedup. Hyperparameters
are labeled above each bar (consistent with the speedup chart).

In [ ]:
mdf_mat = mdf_best[mdf_best["mat"] > 0].copy()
fig, ax = plt.subplots(figsize=(max(11, len(mdf_mat) * 1.05), 7))
x = np.arange(len(mdf_mat))
ax.bar(x, mdf_mat["mat"].values, color=mdf_mat["color"].values,
       edgecolor="black", linewidth=0.4)
for i, r in mdf_mat.iterrows():
    mat = r["mat"]
    marker = " ★" if r["family"] == "extension_oracle" else ""
    ax.text(i, mat + 0.05, f"{mat:.2f}{marker}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
    sub = bar_sub_label(r["family"], r["hp"], r["best_budget"], r.get("sk_tag"))
    if sub:
        ax.text(i, mat + 0.30, sub, ha="center", va="bottom",
                fontsize=7, color="#444")

# Gap arrow on MAT (oracle vs hybrid)
fams = list(mdf_mat["family"].values)
mats = list(mdf_mat["mat"].values)
ext_or_idx = next((i for i, f in enumerate(fams) if f == "extension_oracle"), None)
hyb_picks = [(mats[i], i) for i, f in enumerate(fams) if f in HYB_FAMS]
if ext_or_idx is not None and hyb_picks:
    h_mat, h_idx = max(hyb_picks)
    e_mat = mats[ext_or_idx]
    pct = (e_mat / h_mat - 1) * 100
    x_lo, x_hi = sorted([h_idx, ext_or_idx])
    ax.hlines(h_mat, x_lo - 0.4, x_hi + 0.4, colors="red", linestyles="--", alpha=0.8, linewidth=1.5)
    ax.hlines(e_mat, x_lo - 0.4, x_hi + 0.4, colors="red", linestyles="--", alpha=0.8, linewidth=1.5)
    arrow_x = x_hi + 0.6
    ax.annotate("", xy=(arrow_x, e_mat), xytext=(arrow_x, h_mat),
                arrowprops=dict(arrowstyle="<->", color="red", lw=2))
    ax.text(arrow_x + 0.15, (h_mat + e_mat) / 2, f"{pct:+.1f}%",
            ha="left", va="center", color="red", fontsize=11, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(mdf_mat["pretty"], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("MAT (Mean Accepted Tokens / step)")
ax.set_title(f"{WORKLOAD} — MAT at speedup-best variant per method")
ax.grid(axis="y", alpha=0.2)
ax.set_ylim(top=mdf_mat["mat"].max() * 1.40)
plt.tight_layout(); plt.show()


## 6. Speedup vs MAT scatter

In [ ]:
mdf_scatter = mdf_best[mdf_best["mat"] > 0].copy()
fig, ax = plt.subplots(figsize=(11, 7))
for _, r in mdf_scatter.iterrows():
    edge = "red" if r["family"] == "extension_oracle" else "black"
    lw = 2.0 if r["family"] == "extension_oracle" else 0.7
    ax.scatter(r["mat"], r["speedup_real"], c=r["color"], s=180,
               edgecolors=edge, linewidth=lw, zorder=5)
    label = ("★ " if r["family"] == "extension_oracle" else "") + r["pretty"]
    sub = f"{r['sk_tag']}, B={r['best_budget']}" if r['sk_tag'] != "?" else f"B={r['best_budget']}"
    ax.annotate(f"{label}\n{sub}", (r["mat"], r["speedup_real"]),
                xytext=(8, 4), textcoords="offset points", fontsize=8, alpha=0.85)
ax.axhline(y=1.0, color="red", linestyle=":", alpha=0.3)
ax.set_xlabel("MAT (Mean Accepted Tokens)")
ax.set_ylabel("Real-cost speedup (×)")
ax.set_title(f"{WORKLOAD} — Speedup vs MAT (each method's own optimum)")
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


## 7. Budget sensitivity (top 6 best methods)

In [ ]:
# Top 6 methods by best speedup — each plotted from its OWN winning sim's curve.
top_methods = mdf_best.head(6)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
for _, r in top_methods.iterrows():
    name = r["method"]
    sim = r["sim"]
    df_i = SIM_DFS.get(sim)
    if df_i is None: continue
    col_spd = f"{name}_speedup_real"
    col_mat = f"{name}_mat"
    color = r["color"]
    label = f"{r['pretty']} @ {r['sk_tag']}"
    if col_spd in df_i.columns:
        ax1.plot(df_i["budget"], df_i[col_spd], "o-", label=label,
                 color=color, linewidth=2, markersize=5)
    if col_mat in df_i.columns:
        ax2.plot(df_i["budget"], df_i[col_mat], "s-", label=label,
                 color=color, linewidth=2, markersize=5)
for ax, ylabel, title in [(ax1, "spd_real (×)", "Speedup vs Budget"),
                          (ax2, "MAT", "MAT vs Budget")]:
    ax.set_xlabel("Budget"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_xscale("log", base=2)
    ax.legend(fontsize=7); ax.grid(alpha=0.2)
ax1.axhline(y=1.0, color="red", linestyle=":", alpha=0.3)
plt.suptitle(f"{WORKLOAD} — top-6 methods (each plotted from its own winning sim)", y=1.02)
plt.tight_layout(); plt.show()


## 8. Per-step cost breakdown

Target forward + draft cost at each method's best budget. Vanilla shown for reference.

In [ ]:
# Per-step cost breakdown for representative families
rep_families = ["single:eagle3", "single:suffix", "hybrid_e3",
                "extension", "extension_oracle"]
rep_rows = []
for fam in rep_families:
    rows = mdf_best[mdf_best["family"] == fam]
    if not rows.empty:
        rep_rows.append(rows.iloc[0])

labels, target_ms, draft_ms, budgets, spds = [], [], [], [], []
labels.append("Vanilla\n(no spec)"); target_ms.append(vanilla_ms); draft_ms.append(0)
budgets.append(0); spds.append(1.0)
for r in rep_rows:
    sub = bar_sub_label(r["family"], r["hp"], r["best_budget"], r.get("sk_tag"))
    lbl = r["pretty"] + (f"\n{sub}" if sub else "")
    labels.append(lbl)
    target_ms.append(r["avg_target_ms"]); draft_ms.append(r["avg_draft_ms"])
    budgets.append(r["best_budget"]); spds.append(r["speedup_real"])

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(13, 7))
ax.bar(x, target_ms, color="#1f77b4", edgecolor="black", label="target forward (ms)")
ax.bar(x, draft_ms, bottom=target_ms, color="#ff7f0e", edgecolor="black", label="draft (ms)")
for i, (t, d, B, s) in enumerate(zip(target_ms, draft_ms, budgets, spds)):
    if t > 0:
        ax.text(i, t/2, f"{t:.1f}", ha="center", va="center",
                color="white", fontweight="bold", fontsize=9)
    if d > 0:
        ax.text(i, t+d/2, f"{d:.1f}", ha="center", va="center",
                color="white", fontweight="bold", fontsize=9)
    total = t + d
    if total > 0:
        ax.text(i, total + max(target_ms+draft_ms)*0.02,
                f"step={total:.1f}\nspd={s:.2f}×",
                ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("Per-step latency (ms)")
ax.set_title(f"{WORKLOAD} — per-step cost composition (each method's own optimum)")
ax.legend(loc="upper left"); ax.grid(axis="y", alpha=0.2)
plt.tight_layout(); plt.show()


## 9. Implementation notes — current methods

All methods live in `simulation/evaluation/run_tree_oracle_sim.py`. Cost model:
- `step_cost = target_forward(ext_size) + draft_cost`
- `draft_cost = eagle3_draft(B) + n_actual_suffix_calls × suffix_speculate_ms` (sequential)
- `n_actual_suffix_calls` is dynamically counted (not heuristic)

### Baselines (single proposer)
| method | tree |
|---|---|
| `single:eagle3` | EAGLE3 backbone, top-B by path_prob |
| `single:suffix` | live `SuffixDecodingCache.speculate()` (aggressive params) |
| `single:draft_model` | length-`min(B, 16)` chain from draft LM |

### Hybrid (paper SuffixDecoding)
| method | gate |
|---|---|
| `hybrid_e3:t` | suffix if score ≥ t (paper-faithful F=1.0, T=0.1) else EAGLE3 |
| `hybrid_e3_sfx:F:T:N:t` | NEW — same gate but custom suffix params (F, T, N) |
| `hybrid_dm:t` | suffix if score ≥ t else draft-model chain |

### Extension (eagle3 base + suffix grafts at every node)
| method | accounting |
|---|---|
| `extension` | deployable, ext_size = full tree |
| `extension_oracle` ★ | oracle, ext_size = base + accepted_suffix |
| `extension_sfx:F:T:N` | custom suffix params (deployable) |
| `extension_sfx_oracle:F:T:N` | custom suffix params (oracle) |
| `extension_prune_pt:t` | drop base nodes with path_p_t < t |
| `extension_prune_pt_oracle:t` | same + oracle accounting |
| `extension_hybrid:t` | gate suffix-only / ext fallback (deployable) |
| `extension_hybrid_oracle:t` | same gate + oracle accounting on ext branch |

### Forbidden (per `project_50pct_gap_goal.md`)
| method | reason |
|---|---|
| `extension_oracle_path` | path-only accounting unrealistic (verify pays for whole tree) |
| `extension_hybrid_perfect_oracle` | per-step ORACLE choice between paths is unrealistic |
| `extension_nlevel*` | recursive grafts inflate tree; deployable counterpart impractical |
| `extension_dual_method_oracle*` | same nlevel-based concern |


## 10. Cross-sim method-optimum-vs-method-optimum

Each method picks its own best (s, k, B, threshold) across **all** discovered sims for the
selected workload. This is the FAIRNESS-correct comparison per
`feedback_method_optimum_independent.md`: never pin both methods to the same reslice.

In [ ]:
# Same as the summary table (cell §3), repeated here for clarity:
# every method picked its OWN best (s, k, B, F, T, N, threshold) across all
# discovered sims for the workload. No cell pins methods to a single sim.
print(f"=== {WORKLOAD} — per-method cross-sim optimum ===\n")
print(f"{'method':<35} {'variant':<46} {'reslice':<10} {'B':>4} {'spd_real':>9}")
print("-" * 110)
for _, r in mdf_best.iterrows():
    print(f"{r['pretty']:<35} {r['method'][:44]:<46} {r['sk_tag']:<10} {r['best_budget']:>4} "
          f"{r['speedup_real']:>9.3f}")

# Final headline
ext_or = mdf_best[mdf_best["family"] == "extension_oracle"]
hyb = mdf_best[mdf_best["family"] == "hybrid_e3"]
if len(ext_or) and len(hyb):
    e = ext_or.iloc[0]; h = hyb.iloc[0]
    gap = (e["speedup_real"] / h["speedup_real"] - 1) * 100
    print(f"\n★ HEADLINE GAP: {gap:+.2f}%   (50% target gap = {50 - gap:+.2f}%)")
